# Import thư viện & Cấu hình hệ thống

In [1]:
import re
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

# Cấu hình hiển thị kiểm soát dữ liệu
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
print("Đã thiết lập môi trường xử lý dữ liệu.")

Đã thiết lập môi trường xử lý dữ liệu.


# Khai báo đường dẫn & Nạp dữ liệu gốc

In [2]:
# Định nghĩa đường dẫn động bằng pathlib
DATA_DIR = Path("Dry_Bean_Dataset")
DATA_PATH = Path("Dry_Bean_Dataset.xlsx")

TRAIN_OUTPUT_PATH = Path("dry_bean_train.csv")
TEST_OUTPUT_PATH = Path("dry_bean_test.csv")

# Kiểm tra sự tồn tại của tệp nguồn
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy tệp Excel tại: {DATA_PATH.resolve()}")

# Đọc dữ liệu từ file Excel
df = pd.read_excel(DATA_PATH, engine="openpyxl")
print(f"Nạp dữ liệu thành công. Kích thước ban đầu: {df.shape}")

Nạp dữ liệu thành công. Kích thước ban đầu: (13611, 17)


# Chuẩn hóa theo tên cột (CamelCase sang snake_case)

In [3]:
def clean_column_name(name):
    # Tách các từ viết hoa liền nhau (CamelCase)
    s1 = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
    s2 = re.sub('([a-z0-9])([A-Z])', r'\1_\2', s1)
    # Loại bỏ ký tự đặc biệt, chuyển chữ thường, xóa khoảng trắng thừa
    clean = re.sub(r'[^a-zA-Z0-9]+', '_', s2).lower().strip('_')
    # Sửa lỗi chính tả cố hữu của bộ dữ liệu gốc
    if clean == "aspect_ration":
        clean = "aspect_ratio"
    return clean

# Áp dụng chuẩn hóa cho toàn bộ DataFrame
df.columns = [clean_column_name(col) for col in df.columns]
print("Danh sách tên cột sau khi chuẩn hóa chuẩn cấu trúc:")
print(df.columns.tolist())

Danh sách tên cột sau khi chuẩn hóa chuẩn cấu trúc:
['area', 'perimeter', 'major_axis_length', 'minor_axis_length', 'aspect_ratio', 'eccentricity', 'convex_area', 'equiv_diameter', 'extent', 'solidity', 'roundness', 'compactness', 'shape_factor1', 'shape_factor2', 'shape_factor3', 'shape_factor4', 'class']


# Kiểm tra khuyết thiếu & Xử lý trùng lặp dữ liệu

In [4]:
# 1. Kiểm tra giá trị khuyết thiếu (Missing Values)
total_missing = df.isna().sum().sum()
print(f"Tổng số lượng giá trị thiếu trong bảng: {total_missing}")

# Chỉ thực hiện dropna nếu thực sự phát hiện có giá trị khuyết thiếu
if total_missing > 0:
    df = df.dropna().reset_index(drop=True)
    print("-> Đã loại bỏ các dòng chứa giá trị khuyết thiếu.")

# 2. Kiểm tra và loại bỏ dòng trùng lặp hoàn toàn (Duplicates)
duplicate_count = df.duplicated().sum()
print(f"Số lượng dòng trùng lặp cấu trúc thập phân: {duplicate_count}")

if duplicate_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"-> Kích thước DataFrame sau khi dọn dẹp trùng lặp: {df.shape}")

Tổng số lượng giá trị thiếu trong bảng: 0
Số lượng dòng trùng lặp cấu trúc thập phân: 68
-> Kích thước DataFrame sau khi dọn dẹp trùng lặp: (13543, 17)


# Kiểm tra phân vị để cô lập giá trị ngoại lai (Outliers)

In [5]:
# Đo lường phân phối để phát hiện bất thường hình học (Ví dụ với biến Area)
q1 = df['area'].quantile(0.25)
q3 = df['area'].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = df[(df['area'] < lower_bound) | (df['area'] > upper_bound)]
print(f"Số lượng hạt đậu có diện tích dị thường (Outliers tiềm năng): {len(outliers)} mẫu.")
print("Lưu ý: Không xóa outliers ở đây để tránh mất bối cảnh hình học thực tế khi huấn luyện Cây quyết định/Rừng ngẫu nhiên.")

Số lượng hạt đậu có diện tích dị thường (Outliers tiềm năng): 551 mẫu.
Lưu ý: Không xóa outliers ở đây để tránh mất bối cảnh hình học thực tế khi huấn luyện Cây quyết định/Rừng ngẫu nhiên.


# Phân tách Features/Target & Kiểm tra phân bố lớp

In [6]:
TARGET_COL = "class"
FEATURE_COLS = [col for col in df.columns if col != TARGET_COL]

X = df[FEATURE_COLS].copy()
y = df[TARGET_COL].copy()

print("Tỷ lệ phân bố phân khúc các loại hạt đậu trong tập dữ liệu:")
print(y.value_counts(normalize=True).round(4) * 100)

Tỷ lệ phân bố phân khúc các loại hạt đậu trong tập dữ liệu:
class
DERMASON    26.18
SIRA        19.46
SEKER       14.97
HOROZ       13.73
CALI        12.04
BARBUNYA     9.76
BOMBAY       3.85
Name: proportion, dtype: float64


# Chia tập dữ liệu Train/Test phân tầng (Stratified Split)

In [7]:
# Chia tỷ lệ 80/20, cố định random_state và sử dụng stratify để đồng đều tỷ lệ nhãn
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Tái cấu trúc lại thành DataFrame hoàn chỉnh để chuẩn bị xuất tệp
train_df = X_train.copy()
train_df[TARGET_COL] = y_train

test_df = X_test.copy()
test_df[TARGET_COL] = y_test

print(f"Kích thước tập huấn luyện (Train Set): {train_df.shape}")
print(f"Kích thước tập kiểm thử (Test Set): {test_df.shape}")

Kích thước tập huấn luyện (Train Set): (10834, 17)
Kích thước tập kiểm thử (Test Set): (2709, 17)


# Xuất dữ liệu sạch ra tệp CSV & Xác thực cuối

In [8]:
# Lưu trữ dữ liệu xuống đĩa cứng
train_df.to_csv(TRAIN_OUTPUT_PATH, index=False)
test_df.to_csv(TEST_OUTPUT_PATH, index=False)

print(f"Đã lưu tệp huấn luyện sạch tại: {TRAIN_OUTPUT_PATH.resolve()}")
print(f"Đã lưu tệp kiểm thử sạch tại: {TEST_OUTPUT_PATH.resolve()}")

# Kiểm tra xác thực cấu trúc file sau khi lưu
verify_train = pd.read_csv(TRAIN_OUTPUT_PATH)
print(f"\nXác thực lại tệp vừa ghi thành công. Kích thước Train đọc lại: {verify_train.shape}")

Đã lưu tệp huấn luyện sạch tại: C:\Users\admin\Documents\GitHub\mliot-pyml-2026-hw\week04\Homework_b7\Dry_Bean_Dataset\dry_bean_train.csv
Đã lưu tệp kiểm thử sạch tại: C:\Users\admin\Documents\GitHub\mliot-pyml-2026-hw\week04\Homework_b7\Dry_Bean_Dataset\dry_bean_test.csv

Xác thực lại tệp vừa ghi thành công. Kích thước Train đọc lại: (10834, 17)
